# 01 · Download GoldenCheetah OpenData (task 1.1)

Data: Liversedge, M. (2020) *GoldenCheetah OpenData Project*, OSF, doi:10.17605/OSF.IO/6HFPZ.
One zip per athlete, saved to Drive under `thesis-cycling/data/raw/goldencheetah/zips/`.

Steps, in order. Each is resumable: if Colab disconnects, run the setup cells and the step again.

1. **List** all files on OSF into `manifest.csv` (a few minutes). **Send Claude the printed summary.**
2. **Download a sample** of 100 athletes and index it, to check everything works.
3. **Download everything** (hours; leave it running, or run it overnight on your laptop with the same command).
4. **Index** all zips, which the audit notebook (02) reads.

*Optional:* if OSF answers with many "HTTP 429" (rate limit) messages, make a free OSF account, create a
personal access token (Settings → Personal access tokens, scope `osf.full_read`) and add it as a Colab
secret named `OSF_TOKEN`. The next cell picks it up.

In [ ]:
# --- settings -------------------------------------------------------------
REPO = "Kevin-ai-jpg/BachelorThesis-MachineLearning-RoadCycling"
BRANCH = "main"
DRIVE_DATA = "/content/drive/MyDrive/thesis-cycling/data"
EXTRAS = "track"

In [ ]:
# Same setup as 00_colab_setup: Drive, repo, install, import
from google.colab import drive
drive.mount("/content/drive")
import os, sys, subprocess, pathlib, importlib
pathlib.Path(DRIVE_DATA).mkdir(parents=True, exist_ok=True)
os.environ["CYCLINGML_DATA"] = DRIVE_DATA
os.environ["CYCLINGML_MLRUNS"] = f"{DRIVE_DATA}/mlruns"
token = None
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    pass
url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
repo_dir = "/content/repo"
if os.path.isdir(f"{repo_dir}/.git"):
    subprocess.run(["git", "-C", repo_dir, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", repo_dir, "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, url, repo_dir], check=True)
%cd /content/repo
!git log -1 --oneline
!pip install -q -e ".[{EXTRAS}]"
if "/content/repo/src" not in sys.path:
    sys.path.insert(0, "/content/repo/src")
importlib.invalidate_caches()
from cyclingml import paths
GC = paths.raw_dir("goldencheetah")
GC.mkdir(parents=True, exist_ok=True)
print("GoldenCheetah folder:", GC)

In [ ]:
try:
    from google.colab import userdata
    os.environ["OSF_TOKEN"] = userdata.get("OSF_TOKEN")
    print("Using OSF token")
except Exception:
    print("No OSF token, anonymous access")

## 1 · List files on OSF

In [ ]:
!python -m cyclingml.io.osf list --node 6hfpz --out "{GC}/manifest.csv"

**Send Claude everything the cell above printed** (file counts, sizes, folders).

## 2 · Download a sample of 100 athletes

In [ ]:
!python -m cyclingml.io.osf download --manifest "{GC}/manifest.csv" --dest "{GC}/zips" --limit 100 --workers 4

In [ ]:
!python -m cyclingml.io.goldencheetah index --zips "{GC}/zips" --out "{GC}/index" --workers 2

## 3 · Download everything

Downloads all zips in the manifest; files already complete are skipped. Failures are written to
`zips/failed.txt`; just run the cell again to retry them. To run it on your laptop instead
(overnight), use the same command in a terminal from the repo folder, with `CYCLINGML_DATA` set to
your data folder.

In [ ]:
!python -m cyclingml.io.osf download --manifest "{GC}/manifest.csv" --dest "{GC}/zips" --workers 4

## 4 · Index all zips

In [ ]:
!python -m cyclingml.io.goldencheetah index --zips "{GC}/zips" --out "{GC}/index" --workers 2
!du -sh "{GC}/zips"; ls "{GC}/zips" | wc -l

Then open `02_data_audit.ipynb`.